# local_lora_train

ローカルdatasetから役割別LoRAを学習し、各キャラフォルダ直下へ保存します。

生成は `local_character_single.ipynb` で行います。

保存例:

```text
./<dataset>/<character>/<character>-base.safetensors
./<dataset>/<character>/<character>-face.safetensors
./<dataset>/<character>/<character>-look.safetensors
./<dataset>/<character>/<character>-fullbody.safetensors
./<dataset>/<character>/<character>-style.safetensors
./<dataset>/<character>/<character>-outfit.safetensors
```

## Jetson Orin Nanoで長時間バックグラウンド実行

Jetsonではブラウザ上で実行し続けるより、`Docker + tmux + nbconvert` でNotebookを非対話実行する方が安定します。
コンテナはプロジェクトrootを `/workspace` にbindするので、学習済みLoRAはホスト側の各キャラフォルダ直下に残ります。

プロジェクトrootでコンテナを起動:

```bash
bash ./docker/run_l4t.sh
```

別ターミナルでtmuxを開始:

```bash
tmux new -s lora
```

tmux内でコンテナに入る:

```bash
docker exec -it ai-image-lab-l4t bash
```

コンテナ内でNotebookを最後まで実行:

```bash
cd /workspace
python3 -m jupyter nbconvert \
  --to notebook \
  --execute local_lora_train.ipynb \
  --output local_lora_train.executed.ipynb \
  --ExecutePreprocessor.timeout=-1
```

tmuxから切り離し:

```text
Ctrl-b d
```

再接続:

```bash
tmux attach -t lora
```

既に存在する `.safetensors` はスキップするため、中断後に同じNotebookを再実行しても続きから進めやすい構成です。


## 編集する設定

In [ ]:
from pathlib import Path

DATASET_ROOT = Path("./dataset")  # 実データセットrootを指定
MODEL_ID = "stablediffusionapi/counterfeit-v30"
IS_SDXL = False

TARGET_CHARAS = []  # [] なら全キャラ。絞る場合はキャラフォルダ名を指定
TRAIN_ROLES = ["base", "face", "look", "fullbody", "style", "outfit"]

TRAIN_RESOLUTION = 512
BATCH_SIZE = 1
MAX_TRAIN_STEPS = 1200
LOCAL_TEST_TRAIN_STEPS = 1
LORA_RANK = 32
LORA_ALPHA = 32
LEARNING_RATE = 1e-4
TRAIN_SEED = 42
SAVE_EVERY_N_STEPS = 0

USE_8BIT_ADAM = False
REQUIRE_CUDA_FOR_FULL_TRAINING = True

WORK_DIR = Path("ai-image-lab-work")
SD_SCRIPTS_DIR = WORK_DIR / "sd-scripts"
OUTPUT_DIR = WORK_DIR / "output" / "local_lora_train"


## 通常編集不要: 環境確認

In [ ]:
import gc
import importlib
import os
import subprocess
import shutil
import sys
from pathlib import Path


def _pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)


def ensure_installed(import_name, pkgs=None):
    try:
        importlib.import_module(import_name)
        return
    except ImportError:
        pass
    _pip_install(*(pkgs or [import_name]))


_pip_install(
    "transformers==4.54.1",
    "diffusers[torch]==0.32.1",
    "accelerate==1.6.0",
    "huggingface-hub==0.34.3",
    "safetensors==0.4.5",
    "toml",
    "imagesize",
    "ftfy",
    "einops",
    "voluptuous",
    "tensorboard",
)
ensure_installed("torch", ["torch", "torchvision"])
if USE_8BIT_ADAM:
    try:
        _pip_install("bitsandbytes")
    except Exception as exc:
        print("bitsandbytes install skipped:", exc)

import matplotlib.pyplot as plt
import toml
import torch
from PIL import Image

if torch.cuda.is_available():
    DEVICE = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

WORK_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
IMG_EXTS = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("DEVICE:", DEVICE)


## 通常編集不要: キャラ検出と学習対象

In [ ]:
ROLE_SOURCES = {
    "base": ["lora"],
    "face": ["face"],
    "look": ["portrait", "illust", "anime"],
    "fullbody": ["portrait", "game", "lora"],
    "style": ["illust"],
    "outfit": ["outfit", "costume"],
}


def list_images(path):
    path = Path(path)
    if not path.exists():
        return []
    return sorted([p for p in path.rglob("*") if p.suffix.lower() in IMG_EXTS])


def find_chara_dirs():
    found = {}
    for p in DATASET_ROOT.iterdir():
        if not p.is_dir():
            continue
        names = {x.name for x in p.iterdir() if x.is_dir()}
        if names & {"lora", "portrait", "illust", "anime", "game", "face"}:
            found[p.name] = p
    return found


chara_dirs = find_chara_dirs()
if TARGET_CHARAS:
    chara_dirs = {k: v for k, v in chara_dirs.items() if k in TARGET_CHARAS}

print("学習対象キャラ:", list(chara_dirs))
for chara, root in chara_dirs.items():
    print("\n", chara, root)
    for role in TRAIN_ROLES:
        sources = ROLE_SOURCES.get(role, [])
        count = sum(len(list_images(root / s)) for s in sources)
        print(f"  {role}: {count} images from {sources}")


## 通常編集不要: プレビュー

In [ ]:

preview = []
for chara, root in chara_dirs.items():
    for role in TRAIN_ROLES:
        for source in ROLE_SOURCES.get(role, []):
            imgs = list_images(root / source)
            if imgs:
                preview.append((f"{chara}/{role}/{source}", imgs[0]))
            if len(preview) >= 6:
                break
        if len(preview) >= 6:
            break
    if len(preview) >= 6:
        break

if preview:
    fig, axes = plt.subplots(1, len(preview), figsize=(4 * len(preview), 4))
    if len(preview) == 1:
        axes = [axes]
    for ax, (label, path) in zip(axes, preview):
        ax.imshow(Image.open(path).convert("RGB"))
        ax.set_title(label)
        ax.axis("off")
    plt.show()

print("caption txt preview:")
shown = 0
for chara, root in chara_dirs.items():
    for role in TRAIN_ROLES:
        for source in ROLE_SOURCES.get(role, []):
            for img in list_images(root / source):
                txt = img.with_suffix(".txt")
                if txt.exists():
                    print(f"\n--- {chara}/{role}/{source}: {txt.name}")
                    print(txt.read_text(encoding="utf-8", errors="replace")[:800])
                    shown += 1
                if shown >= 6:
                    break
            if shown >= 6:
                break
        if shown >= 6:
            break
    if shown >= 6:
        break
if shown == 0:
    print("対応する .txt caption は見つかりませんでした。caption_extension=.txt で学習するため、必要なら画像と同名txtを追加してください。")


## 通常編集不要: sd-scripts取得

In [ ]:
if not (SD_SCRIPTS_DIR / "train_network.py").exists():
    if SD_SCRIPTS_DIR.exists():
        shutil.rmtree(SD_SCRIPTS_DIR)
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/kohya-ss/sd-scripts.git", str(SD_SCRIPTS_DIR)], check=True)
print("SD_SCRIPTS_DIR =", SD_SCRIPTS_DIR)


## 通常編集不要: role別LoRA学習

In [ ]:
def optimizer_type():
    if DEVICE == "cuda" and USE_8BIT_ADAM:
        try:
            import bitsandbytes  # noqa: F401
            return "AdamW8bit"
        except Exception as exc:
            print("bitsandbytes unavailable. AdamWへfallback:", exc)
    return "AdamW"


def precision_settings():
    if DEVICE == "cuda":
        return "fp16", "fp16"
    return "no", "float"


def build_dataset_config(chara_root, role):
    subsets = []
    for source in ROLE_SOURCES.get(role, []):
        image_dir = chara_root / source
        if list_images(image_dir):
            subsets.append({"image_dir": str(image_dir.resolve()), "num_repeats": 1})
    if not subsets:
        return None
    return {
        "general": {"shuffle_caption": True, "caption_extension": ".txt", "keep_tokens": 1},
        "datasets": [
            {
                "resolution": TRAIN_RESOLUTION,
                "batch_size": BATCH_SIZE,
                "enable_bucket": True,
                "min_bucket_reso": 256,
                "max_bucket_reso": 1024,
                "bucket_reso_steps": 64,
                "subsets": subsets,
            }
        ],
    }


def stream_command(cmd, env):
    print("実行コマンド:", " ".join(str(x) for x in cmd))
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
    code = proc.wait()
    if code != 0:
        raise subprocess.CalledProcessError(code, cmd)


trained_loras = {}
opt = optimizer_type()
mixed_precision, save_precision = precision_settings()

if REQUIRE_CUDA_FOR_FULL_TRAINING and DEVICE != "cuda" and MAX_TRAIN_STEPS > LOCAL_TEST_TRAIN_STEPS:
    raise RuntimeError("CUDA以外で重いLoRA学習を開始しません。MAX_TRAIN_STEPSを下げるかCUDA環境で実行してください。")

for chara, root in chara_dirs.items():
    trained_loras[chara] = {}
    for role in TRAIN_ROLES:
        dataset_config = build_dataset_config(root, role)
        if dataset_config is None:
            print(f"[{chara}-{role}] 学習画像なし -> スキップ")
            continue
        images_count = sum(len(list_images(Path(s["image_dir"]))) for s in dataset_config["datasets"][0]["subsets"])
        steps = MAX_TRAIN_STEPS if DEVICE == "cuda" else min(MAX_TRAIN_STEPS, LOCAL_TEST_TRAIN_STEPS)
        output_name = f"{chara}-{role}"
        output_path = root / f"{output_name}.safetensors"
        if output_path.exists():
            print(f"[{output_name}] 既存 -> スキップ: {output_path}")
            trained_loras[chara][role] = output_path
            continue

        toml_path = OUTPUT_DIR / f"_train_{output_name}.toml"
        with open(toml_path, "w", encoding="utf-8") as f:
            toml.dump(dataset_config, f)

        print(f"\n[{output_name}] images={images_count}, steps={steps}, output={output_path}")
        train_script = "sdxl_train_network.py" if IS_SDXL else "train_network.py"
        cmd = [
            sys.executable, "-m", "accelerate.commands.launch",
            "--num_processes", "1",
            "--num_machines", "1",
            "--mixed_precision", mixed_precision,
            "--dynamo_backend", "no",
            "--num_cpu_threads_per_process", "1",
            str(SD_SCRIPTS_DIR / train_script),
            "--pretrained_model_name_or_path", MODEL_ID,
            "--dataset_config", str(toml_path),
            "--output_dir", str(root),
            "--output_name", output_name,
            "--save_model_as", "safetensors",
            "--network_module", "networks.lora",
            "--network_dim", str(LORA_RANK),
            "--network_alpha", str(LORA_ALPHA),
            "--learning_rate", str(LEARNING_RATE),
            "--lr_scheduler", "cosine_with_restarts",
            "--max_train_steps", str(steps),
            "--mixed_precision", mixed_precision,
            "--save_precision", save_precision,
            "--optimizer_type", opt,
            "--clip_skip", "1" if IS_SDXL else "2",
            "--seed", str(TRAIN_SEED),
            "--cache_latents",
            "--gradient_checkpointing",
            "--logging_dir", str(OUTPUT_DIR / "logs"),
        ]
        if SAVE_EVERY_N_STEPS and int(SAVE_EVERY_N_STEPS) > 0:
            cmd.extend(["--save_every_n_steps", str(SAVE_EVERY_N_STEPS)])
        env = os.environ.copy()
        env["PYTHONUNBUFFERED"] = "1"
        try:
            stream_command(cmd, env)
        except subprocess.CalledProcessError:
            if opt == "AdamW8bit":
                print("AdamW8bit失敗。AdamWで再試行します。")
                cmd[cmd.index("--optimizer_type") + 1] = "AdamW"
                stream_command(cmd, env)
            else:
                raise
        if not output_path.exists():
            raise RuntimeError(f"学習後のLoRAが見つかりません: {output_path}")
        trained_loras[chara][role] = output_path
        print(f"[{output_name}] done: {output_path}")
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print("\ntrained_loras:")
print(json.dumps({c: {r: str(p) for r, p in roles.items()} for c, roles in trained_loras.items()}, ensure_ascii=False, indent=2))


## 通常編集不要: LoRAメタデータ確認

In [ ]:
from safetensors import safe_open

for chara, roles in trained_loras.items():
    for role, path in roles.items():
        path = Path(path)
        if not path.exists():
            continue
        print(f"\n=== {chara}/{role}: {path}")
        with safe_open(path, framework="pt", device="cpu") as f:
            md = f.metadata() or {}
        print("size_mb:", round(path.stat().st_size / 1024 / 1024, 2))
        for key in [
            "ss_output_name", "ss_sd_model_name", "ss_steps", "ss_epoch",
            "ss_num_epochs", "ss_max_train_steps", "ss_network_dim",
            "ss_learning_rate", "ss_optimizer", "ss_mixed_precision",
            "ss_dataset_dirs",
        ]:
            if key in md:
                print(f"{key}: {md[key]}")
